# 05. 기본 회귀분석 — 원래 연구 설계 그대로

**원본**: `Analisis3/004_regression_…ipynb`, `분석_구조_정리.md`의 모형 계획

원래 설계(`README_친구에게.txt`)의 모형을 정리본 데이터로 다시 돌린다. 모형 형태는 바꾸지 않았다.

| 모형 | Y | X | 방법 |
|---|---|---|---|
| 본모형 | 07~09시 승차 증감(명) | 보행시간, IC, 경쟁 목적지 비율, 20~50대 인구, 건축물 수 | OLS + HC3 |
| 대안모형 | 〃 | 20~50대 인구 대신 고령인구 | OLS + HC3 |
| 보조 1 | 감소 여부 (1=감소) | 본모형 X | 로지스틱 |
| 보조 2 | 감소 셀의 감소규모(명) | 본모형 X | OLS + HC3 |

→ 결과가 약한 이유와 새로 시도한 모형은 `2_추가분석`에 있다.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # 정리본 최상위의 config.py

from config import *
setup_notebook()

In [2]:
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

data = load_model_data()
print(f"표본 {len(data)}개 셀 | 07~09시 {data.board_2024.sum():,.0f} → {data.board_2025.sum():,.0f}명 "
      f"({(data.board_2025.sum()/data.board_2024.sum()-1)*100:.1f}%) | 감소 셀 {data.decreased.sum()}개")
data[["change"] + X_MAIN].describe().T.round(2)

표본 132개 셀 | 07~09시 6,044 → 4,757명 (-21.3%) | 감소 셀 78개


,count,mean,std,min,25%,50%,75%,max
change,132.0,-9.75,31.37,-145.00,-14.00,-3.00,2.00,119.00
walk_min,132.0,67.07,53.08,3.55,31.94,48.72,88.20,259.18
ic_access,132.0,5.02,2.66,0.32,2.79,4.66,6.21,11.98
competing_share,132.0,13.96,25.34,0.00,0.00,0.00,18.18,100.00
pop_20_50,132.0,648.23,554.78,0.00,101.75,551.00,1088.25,2141.00
buildings,132.0,32.75,31.27,0.00,13.00,23.00,38.50,187.00


## 본모형 — OLS + HC3

In [3]:
main = sm.OLS(data.change, sm.add_constant(data[X_MAIN])).fit(cov_type="HC3")
print(f"R2 = {main.rsquared:.3f}, adj R2 = {main.rsquared_adj:.3f}, F p-value = {main.f_pvalue:.4f}")
coef_table(main)

R2 = 0.075, adj R2 = 0.038, F p-value = 0.0137


,coef,p,유의
const,-15.3164,0.0249,**
GTX역 보행시간(분),0.1873,0.0011,***
IC까지 도로거리(km),-1.8037,0.0516,*
GTX 경쟁 목적지 비율(%),-0.0993,0.2529,
2024년 20~50대 인구,0.0033,0.4922,
2024년 건축물 수,0.0403,0.5648,


In [4]:
X = sm.add_constant(data[X_MAIN])
vif = pd.DataFrame({"VIF": [variance_inflation_factor(X.values, i) for i in range(1, X.shape[1])]},
                   index=[VARIABLE_LABELS[c] for c in X_MAIN])
display(vif.round(2))
corr = data[X_MAIN + ["pop_elderly"]].corr().round(2)
corr.index = corr.columns = [VARIABLE_LABELS[c] for c in corr.columns]
corr

,VIF
GTX역 보행시간(분),1.70
IC까지 도로거리(km),1.37
GTX 경쟁 목적지 비율(%),1.25
2024년 20~50대 인구,1.24
2024년 건축물 수,1.12


,GTX역 보행시간(분),IC까지 도로거리(km),GTX 경쟁 목적지 비율(%),2024년 20~50대 인구,2024년 건축물 수,2024년 고령인구
GTX역 보행시간(분),1.00,0.42,0.38,-0.41,0.20,-0.28
IC까지 도로거리(km),0.42,1.00,0.33,-0.06,-0.12,-0.01
GTX 경쟁 목적지 비율(%),0.38,0.33,1.00,-0.05,0.15,0.03
2024년 20~50대 인구,-0.41,-0.06,-0.05,1.00,-0.08,0.83
2024년 건축물 수,0.20,-0.12,0.15,-0.08,1.00,0.01
2024년 고령인구,-0.28,-0.01,0.03,0.83,0.01,1.00


→ VIF는 모두 2 미만이라 다중공선성 문제는 없다. 20~50대 인구와 고령인구는 상관이 높아서(0.8 이상) 같은 모형에 넣지 않는다.

## 대안모형 — 20~50대 인구 대신 고령인구

In [5]:
x_alt = ["walk_min", "ic_access", "competing_share", "pop_elderly", "buildings"]
alt = sm.OLS(data.change, sm.add_constant(data[x_alt])).fit(cov_type="HC3")
print(f"R2 = {alt.rsquared:.3f}, adj R2 = {alt.rsquared_adj:.3f}")
coef_table(alt)

R2 = 0.096, adj R2 = 0.060


,coef,p,유의
const,-18.9487,0.0046,***
GTX역 보행시간(분),0.2077,0.0006,***
IC까지 도로거리(km),-1.9558,0.0411,**
GTX 경쟁 목적지 비율(%),-0.1173,0.1825,
2024년 고령인구,0.0287,0.0727,*
2024년 건축물 수,0.0284,0.6934,


## 보조 1 — 감소 여부 로지스틱

In [6]:
logit = sm.Logit(data.decreased, sm.add_constant(data[X_MAIN])).fit(disp=0, cov_type="HC1")
print(f"pseudo R2 = {logit.prsquared:.3f}")
coef_table(logit)

pseudo R2 = 0.136


,coef,p,유의
const,1.4211,0.0168,**
GTX역 보행시간(분),-0.0211,0.0005,***
IC까지 도로거리(km),0.1831,0.0546,*
GTX 경쟁 목적지 비율(%),0.0040,0.6627,
2024년 20~50대 인구,-0.0004,0.3270,
2024년 건축물 수,-0.0120,0.0649,*


## 보조 2 — 감소 셀의 감소규모

In [7]:
decreased = data[data.decreased == 1].assign(decrease=lambda d: -d.change)
scale = sm.OLS(decreased.decrease, sm.add_constant(decreased[X_MAIN])).fit(cov_type="HC3")
print(f"감소 셀 {len(decreased)}개 | R2 = {scale.rsquared:.3f}")
coef_table(scale)

감소 셀 78개 | R2 = 0.138


,coef,p,유의
const,35.3057,0.0001,***
GTX역 보행시간(분),-0.3330,0.0010,***
IC까지 도로거리(km),1.2357,0.3025,
GTX 경쟁 목적지 비율(%),0.2375,0.1505,
2024년 20~50대 인구,-0.0014,0.8164,
2024년 건축물 수,-0.0844,0.4237,


## 요약

- 본모형 R²는 0.07 수준이고, 유의한 변수는 **GTX 보행시간(+)** 정도다. IC까지 도로거리는 경계선(p≈0.05)이다.
- 보행시간이 +라는 것은 '역에서 멀수록 증감이 크다(덜 줄었다)'는 뜻이다. 하지만 이 결과는 **셀 규모 효과와 섞여 있다**(2_추가분석/01 참고).
- 원래 설계 그대로는 설명력이 낮다. 원인 진단과 대안 모형은 **`2_추가분석`**에 있다.